In [ ]:
import pyActigraphy
import plotly.graph_objects as go
import os
import pandas as pd
import numpy as np
import os

In [ ]:
fpath = 'C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara'
# fpath = 'C:\\Users\\gianl\\Desktop\\Actigraphy Sara'

In [ ]:
raw = pyActigraphy.io.read_raw_mtn(fpath + '\\1.0_Joined_23-09-24_03-02-25.mtn')

In [ ]:
raw.name

In [ ]:
raw.start_time

In [ ]:
raw.duration()

In [ ]:
raw.frequency

VISUALISING

In [ ]:
layout = go.Layout(
    title="Actigraphy data",
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Counts/period"),
    showlegend=False
)
go.Figure(data=[go.Scatter(x=raw.data.index.astype(str), y=raw.data)], layout=layout)

MASKING INACTIVE DATA

In [ ]:
raw.light.create_light_mask()

In [ ]:
# manually adding mask to more than one period
periods = [{'start': '2024-10-13 09:00:00', 'stop': '2024-10-13 20:30:00'},{'start': '2024-11-18 14:00:00', 'stop': '2024-11-18 15:00:00'},]

for period in periods:
    raw.add_mask_period(start=period['start'], stop=period['stop'])

In [ ]:
# visualising mask:
go.Figure(data=go.Scatter(x=raw.mask.index.astype(str),y=raw.mask),layout=layout)

In [ ]:
# applying the mask:
raw.light.apply_mask = True

DAILY ACTIVITY PROFILE (create one for UK and one for Italy?)

In [ ]:
layout.update(title="Daily activity profile",xaxis=dict(title="Date time"), showlegend=False);

In [ ]:
daily_profile = raw.average_daily_activity(freq='15min', cyclic=False, binarize=False)

In [ ]:
go.Figure(data=[
    go.Scatter(x=daily_profile.index.astype(str), y=daily_profile)
], layout=layout)

In [ ]:
# Activity onset:
raw.AonT(freq='15min', binarize=True)

In [ ]:
# Activity offset:
raw.AoffT(freq='15min', binarize=True)

DETECTING SLEEP PERIODS

In [ ]:
# all algorithms return a binary time serie

In [ ]:
layout = go.Layout(title="Rest/Activity detection",xaxis=dict(title="Date time"), yaxis=dict(title="Counts/period"), showlegend=False)

In [ ]:
roenneberg = raw.Roenneberg()
# roenneberg_thr = raw.Roenneberg(threshold=0.25, min_seed_period='15min') # TRESHOLD = fraction of the trend to use as a treshold for sleep/wake categorisation. min_seed_period = minimum time period required to identify a potential sleep onset

Roenneberg algorithm = detects consolidated periods of similar activity patterns. It's treshold-based but uses correlations with test series of various lengths to find the consolidated period that best matches the data

In [ ]:
crespo = raw.Crespo()

In [ ]:
aot = raw.Crespo_AoT()
aot

In [ ]:
# create a df with the onset and offset of the activity period scored by the Crespo algorithm 
sleep_periodsX = pd.DataFrame({'start': aot[0], 'stop': aot[1]})

In [ ]:
sleep_periodsX = sleep_periodsX[['stop', 'start']]
sleep_periodsX.rename(columns={'start': 'end_datetime', 'stop': 'start_datetime'}, inplace=True)
sleep_periodsX

COLE-KRIPKE ALGORITHM = epoch-by-epoch rest-activity scoring. Uses a rolling window over the data and convolute them with a "gaussian"-like kernel. If the resulting data is above a predefined treshold, classify as activity

In [ ]:
CK = raw.CK()

In [ ]:
#layout.update(yaxis2=dict(title='Classification',overlaying='y',side='right'), showlegend=True);
#go.Figure(data=[
#    go.Scatter(x=raw.data.index.astype(str),y=raw.data, name='Data'),
#    go.Scatter(x=CK.index.astype(str),y=CK, yaxis='y2', name='CK')
#], layout=layout)

In [ ]:
# dataframe to store the CK sleep scoring
df_CK = pd.DataFrame({'datetime': CK.index, 'scoring': CK})

df_CK = df_CK.reset_index(drop=True)

In [ ]:
# list to store the wake counts
wake_counts = []

# function to count the number of wake periods in a sleep period
for _, row in sleep_periodsX.iterrows():
    start, end = row['start_datetime'], row['end_datetime']
    
    # filter the data to the sleep period
    sleep_period = df_CK[(df_CK['datetime'] >= start) & (df_CK['datetime'] <= end)]
    
    wake_count = sleep_period[sleep_period['scoring'] == 0].shape[0]
    
    wake_counts.append({'start_datetime': start, 'end_datetime': end, 'epochs_awake_count': wake_count})

wake_counts_df = pd.DataFrame(wake_counts)

wake_counts_df.head()

In [ ]:
# wake duration in minutes
wake_counts_df['WASO'] = wake_counts_df['epochs_awake_count']

# remove epochs_awake_count column
wake_counts_df = wake_counts_df.drop(columns='epochs_awake_count')

wake_counts_df.head()

VISUALISING LIGHT DATA

In [ ]:
raw.light

In [ ]:
raw.light.get_channel_list()

In [ ]:
# LIGHT + ACTIVITY
layout = go.Layout(
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Activity counts/period"),
    yaxis2=dict(title='Light intensity',overlaying='y',side='right'),
    showlegend=True
)

fig1 = go.Figure([
    go.Scatter(
        x=raw.data.index.astype(str),
        y=raw.data,
        name='Activity'),
    go.Scatter(
        x=raw.light.get_channel('whitelight').index.astype(str),
        y=raw.light.get_channel('whitelight'),
        yaxis='y2', opacity=0.5,
        name='Light')
], layout=layout)

fig1.show()